# A2 | 09 — Auditar qualidade, datas e elegibilidade

**Pessoa.** Distinguir problema de qualidade, ausência legítima, variável inadequada e risco de vazamento.

**Saída até 10/10:** relatório de qualidade revisado, flags aprovadas e matriz de elegibilidade v1.0.
**Revisão:** B verifica padrões visuais; C transforma as decisões em validações automáticas. **Gate 1 (09/10):** regras de preparação.

In [ ]:
# ----- Setup (não editar) -----
import os, sys
from pathlib import Path

# PKG_ROOT: pasta que contém src/protocolo (onde o pacote foi copiado)
# REPO_ROOT: raiz do repositório (tem .git ou data/); é de onde lemos dados e gravamos saídas
_parents = [Path.cwd(), *Path.cwd().parents]
PKG_ROOT = next((p for p in _parents if (p / "src" / "protocolo").exists()), None)
if PKG_ROOT is None:
    raise FileNotFoundError(
        "Pasta src/protocolo não encontrada acima de " + str(Path.cwd()) + ". "
        "Copie 'src/protocolo/' do pacote para o 'src/' do repositório "
        "(deve existir src/protocolo/verificacoes.py) e rode a célula de novo.")
REPO_ROOT = next((p for p in _parents if (p / ".git").exists() or (p / "data").exists()), PKG_ROOT)
os.chdir(REPO_ROOT); sys.path.insert(0, str(PKG_ROOT))

# Proteção contra outro pacote chamado 'src' (ex.: instalado por pip) ou 'src' antigo em cache no kernel:
# um pacote regular em qualquer ponto do sys.path vence um 'src' sem __init__.py.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]
_init = PKG_ROOT / "src" / "__init__.py"
if not _init.exists():
    _init.touch()
    print("Criado", _init, "(torna o src do projeto um pacote regular)")

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from src.protocolo import verificacoes as v
if getattr(v, "MODULE_VERSION", "0") < "0.8":
    raise RuntimeError(
        "src/protocolo/verificacoes.py está desatualizado (versão " + str(getattr(v, "MODULE_VERSION", "antiga")) + "). "
        "Substitua pelo arquivo do pacote novo e REINICIE o kernel (o Python mantém o módulo antigo em cache).")

# ----- Parâmetros -----
USAR_DEMO = False                       # True = dados sintéticos só para testar o notebook
DATA_PATH = None                        # None = procura Parquet/CSV em data/ ; ou informe o caminho do arquivo
SAMPLE = 0                              # ex.: 200_000 para execução rápida
OUT = "outputs/tables/pessoa"         # saídas (tabelas, atas, decisões), relativas à raiz do repositório
CFG_PATH = next((str(p) for p in [REPO_ROOT / "configs" / "decisoes.yaml", PKG_ROOT / "configs" / "decisoes.yaml"]
                 if p.exists()), "configs/decisoes.yaml")
SUPERVISED_CONFIG = "configs/supervised.yaml"   # config do C (alvo, features, coluna_tempo)
COLUNAS = "classificadas"                   # "minimas" (menos memória) | "classificadas" | "todas" (pesado!) | lista de nomes
EXCLUIR_PROIBIDAS = True        # v1.0: nenhuma coluna proibida segue para o C
print("Raiz do repositório:", REPO_ROOT)
print("Pacote (src/protocolo):", PKG_ROOT)
print("Config de decisões:", CFG_PATH)
cfg = v.load_config(CFG_PATH)
rep = v.Report(OUT, "relatorio_A2")
df = v.prepare(v.carregar_base(DATA_PATH, USAR_DEMO, SAMPLE, COLUNAS, cfg, SUPERVISED_CONFIG))
print(f"Memória do DataFrame: {v.uso_memoria_gb(df):.2f} GB")

## 1. Vazios, ignorados e campos inexistentes por ano

In [ ]:
v.a2_missing(df, cfg, rep)
display(pd.read_csv(Path(OUT) / "a2_campos_inexistentes_por_ano.csv").head(40))

## 2. Coerência de datas e idade
Classifique cada inconsistência: **excluir**, **transformar em ausente**, **manter com flag** ou **apenas documentar** (a coluna `acao_sugerida` é uma proposta).

In [ ]:
v.a2_dates(df, cfg, rep)
display(pd.read_csv(Path(OUT) / "a2_coerencia_datas.csv").head(40))

## 3. Duplicidades
`NU_NOTIFIC` repetido **não** implica mesma pessoa.

In [ ]:
v.a2_duplicates(df, cfg, rep)
display(pd.read_csv(Path(OUT) / "a2_duplicidades.csv").head(40))

## 4. Maturidade de EVOLUCAO por mês de internação
Meses recentes do holdout podem ter evolução ainda aberta.

In [ ]:
v.a2_maturity(df, cfg, rep)
display(pd.read_csv(Path(OUT) / 'a2_maturidade_evolucao_por_mes.csv').tail(24))

## 5. Matriz de elegibilidade
Classes: **incluir**, **incluir com condição**, **auditoria**, **excluir**. Para cada variável responda: *quando foi observada, quando chegou à base e se poderia existir na admissão.* Preencha as colunas em branco do CSV gerado (`a2_matriz_elegibilidade.csv`).

In [ ]:
v.a2_eligibility(df, cfg, rep)
display(pd.read_csv(Path(OUT) / "a2_matriz_elegibilidade.csv").head(40))

## 6. Tabela de decisões

In [ ]:
E = "A2"
v.log_decision(OUT, "D-06", "Ação por tipo de inconsistência de datas",
               "; ".join(f"{r[2]} -> {r[3]}" for r in cfg["date_rules"]), E,
               impacto="Ver a2_coerencia_datas.csv")
v.log_decision(OUT, "D-07", "Duplicidades", "Linhas exatas removidas; NU_NOTIFIC repetido apenas documentado", E,
               impacto="Ver a2_duplicidades.csv")
v.log_decision(OUT, "D-08", "Maturidade de EVOLUCAO",
               f"Meses do holdout com < {cfg['maturity_min_closed_pct']}% de evolução fechada ficam fora ou sob sensibilidade", E,
               impacto="Ver a2_maturidade_evolucao_por_mes.csv")
v.log_decision(OUT, "D-09", "Variáveis proibidas/condicionais",
               "Matriz de elegibilidade v1.0 (classes incluir/condição/auditoria/excluir)", E,
               impacto="C bloqueia automaticamente (test_leakage); B usa na matriz de clustering")
display(pd.read_csv(Path(OUT) / "tabela_decisoes.csv"))

## 7. Exportar a população analítica (v1.0)
Com a matriz de elegibilidade fechada, regrave o dataset do C. Com `EXCLUIR_PROIBIDAS = True`, features classificadas como **excluir** não seguem para o C (o dataset lista o que ficou de fora).

In [ ]:
# Derivar features que o contrato C2 exige (defensivo; só dispara se as bases estiverem carregadas)
if "dias_sintomas_ate_internacao" not in df.columns and {"dt_sin_pri", "dt_interna"}.issubset(df.columns):
    df["dias_sintomas_ate_internacao"] = (df["dt_interna"] - df["dt_sin_pri"]).dt.days
if "sexo_categoria" not in df.columns and "cs_sexo" in df.columns:
    df["sexo_categoria"] = df["cs_sexo"].map({1: "Masculino", 2: "Feminino"}).fillna("Ignorado")

info = v.a1_export_dataset(df, cfg, rep, SUPERVISED_CONFIG, versao="1.0", excluir_proibidas=EXCLUIR_PROIBIDAS)
print("Arquivo:", info["arquivo"], "|", f"{info['n_linhas']:,} linhas")
print("Features ausentes na base:", info["features_ausentes_na_base"] or "nenhuma")
print("Colunas:", info["colunas"])
v.validar_contrato_c2(info["arquivo"], SUPERVISED_CONFIG, rep)   # contrato do C2 (colunas, datas, alvo, leakage)

## 8. Fechamento

In [ ]:
path = rep.write_md()
print("Relatório:", path)
display(pd.DataFrame(rep.findings, columns=["etapa", "verificação", "status", "detalhe"]))

### Checklist de saída
- [ ] Relatório de qualidade revisado e flags aprovadas
- [ ] Matriz de elegibilidade v1.0 preenchida (nenhuma variável em `auditoria`)
- [ ] Lista de variáveis proibidas e condições de saturação, vacinação e geografia fechadas
- [ ] Dataset analítico v1.0 regravado e C avisado
- [ ] Regras entregues a B e C